# RoadAware V3 notebook 22: verified spatial inference and robustness contract

Research serving contract for the ACCEPTED notebook-20 benchmark bundles (`rerun-spatial-model-016-002`, both units `hgb_poisson_fixed::routed_context`,
strong baseline `segment_mean_k365`) and the ACCEPTED notebook-21 colour contracts (`rerun-spatial-eval-017-002`).

* Manager pins (`workflow/serving/input-pins-v3.json`) and the pinned producer manifests are verified BEFORE any notebook source is executed or any
  joblib bundle is deserialized. Missing/tampered inputs give an explicit `artifact_unavailable` service, never a fake zero or a hidden baseline.
* Features come only from the pinned notebook-19 `build_feature_rows`; components, routing and baseline recipe only from the pinned notebook-20
  FUNCTIONS; colours only from the pinned notebook-21 FUNCTIONS. No feature logic is re-implemented here.
* No candidate fit, no refit, no reselection, no network. Baseline sufficient statistics are reconstructed ONCE from the pinned labels with the
  exact notebook-20 benchmark cutoff/recipe and checked against the saved 2024/25 baseline array.
* Output: expected SELECTED+LINKED recorded count (lambda) per segment and horizon. Not an accident probability, not per-vehicle risk.
  2024/25 are previously seen diagnostics; dates after 2025 are research scenarios without observed ground truth. `production_ready` is false.


In [ ]:
# PARAMETERS: roots come from the runner environment (ROADAWARE_BACKEND_ROOT / ROADAWARE_V3_OUTPUT_ROOT); interactive default = this repo's backend/.
import os
from pathlib import Path


def _default_backend(c):
    for p in (c, *c.parents):
        if (p / 'config' / 'spatial_inference_v3.json').is_file():
            return p
        if (p / 'backend' / 'config' / 'spatial_inference_v3.json').is_file():
            return p / 'backend'
    return c


BACKEND = Path(os.environ.get('ROADAWARE_BACKEND_ROOT') or _default_backend(Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('ROADAWARE_V3_OUTPUT_ROOT') or BACKEND).resolve()
CONFIG_PATH = BACKEND / 'config' / 'spatial_inference_v3.json'
print('backend', BACKEND, '| output root', OUTPUT_ROOT)


In [ ]:
# FUNCTIONS (1/5): error codes, hashing, strict JSON, output/rerun guards, the manager-pin trust chain (19/20/21 producer manifests) and the
# hash-verified FUNCTIONS loaders of notebooks 19/20/21. Pure logic; the tests exec these cells. Nothing here fits, downloads or writes outside out dirs.
import datetime as dt
import hashlib
import importlib
import importlib.util
import json
import math
import os
import platform
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd

NL = chr(10)
UNITS = ('daily', '6h')
COMPONENTS = ('core', 'context')
# error codes: request problems vs coverage vs artifact availability are distinct families (never a fake zero)
INVALID_CODES = ('invalid_request', 'invalid_unit', 'invalid_fields', 'invalid_value', 'invalid_date', 'invalid_time_block', 'duplicate_keys',
                 'batch_size', 'date_out_of_range', 'date_before_artifact_asof')
COVERAGE_CODES = ('unknown_segment', 'unsupported_segment')
UNAVAILABLE_CODES = ('artifact_unavailable', 'artifact_missing', 'artifact_hash_mismatch', 'artifact_inconsistent', 'prediction_invalid')


def row_positions(rows, limit=10):
    """Offending request positions as plain ints (first `limit`). Never truth-tests the input: `ndarray or []` is ambiguous for 2+ positions
    and drops a lone row 0. None/list/tuple/ndarray/Index accepted; a boolean array is read as a row mask."""
    if rows is None:
        return []
    a = np.asarray(rows).ravel()
    if a.dtype == bool:
        a = np.flatnonzero(a)
    return [int(r) for r in a[:limit].tolist()]


class ServingError(RuntimeError):
    def __init__(self, code, message, rows=None):
        super().__init__(code + ': ' + message)
        self.code, self.message, self.rows = code, message, row_positions(rows)


def error_family(code):
    return 'invalid' if code in INVALID_CODES else ('coverage' if code in COVERAGE_CODES else 'unavailable')


def utcnow():
    return dt.datetime.now(dt.timezone.utc).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


def jsonable(o):
    # numpy scalars to Python, non-finite floats to None, so json.dumps(allow_nan=False) stays strict
    if isinstance(o, (dict, types.MappingProxyType)):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [jsonable(v) for v in (sorted(o) if isinstance(o, set) else o)]
    if isinstance(o, np.ndarray):
        return [jsonable(v) for v in o.tolist()]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    if o is pd.NA or o is pd.NaT:
        return None
    return o


def strict_json_bytes(obj):
    return (json.dumps(jsonable(obj), indent=2, ensure_ascii=False, allow_nan=False) + NL).encode('utf-8')


def strict_dumps(obj):
    """Compact strict JSON of an API response: refuses NaN/Infinity instead of silently nulling (responses must already be finite)."""
    return json.dumps(obj, ensure_ascii=False, allow_nan=False, separators=(',', ':'))


def read_strict_json(path):
    def bad(c):
        raise ServingError('artifact_inconsistent', 'non-finite JSON constant ' + c + ' in ' + str(path))
    return json.loads(Path(path).read_text('utf-8'), parse_constant=bad)


def atomic_write_bytes(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp-' + str(os.getpid()))
    with open(tmp, 'wb') as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def csv_bytes(df):
    num = df.select_dtypes(include=[np.number])
    if len(num.columns) and np.isinf(num.to_numpy(dtype=float)).any():
        raise ServingError('prediction_invalid', 'refusing to write infinite values to CSV')
    return df.to_csv(index=False, lineterminator=NL).encode('utf-8')


def module_version(name):
    if importlib.util.find_spec(name) is None:
        return None
    return str(getattr(importlib.import_module(name), '__version__', 'unknown'))


def require_modules(pins):
    missing = [n for n in pins if importlib.util.find_spec(n) is None]
    if missing:
        raise ServingError('artifact_unavailable', 'missing Python package(s) ' + ', '.join(missing) + '; ask Codex for ' + ' '.join(pins[n] for n in missing))
    if module_version('sklearn') != '1.6.1':
        raise ServingError('artifact_unavailable', 'scikit-learn ' + str(module_version('sklearn')) + ' != locked 1.6.1 (bundles were fitted with 1.6.1)')
    return {n: module_version(n) for n in pins}


def environment():
    return {'python': platform.python_version(), 'platform': platform.platform(), 'machine': platform.machine(), 'cpu_count': os.cpu_count(),
            'dependencies': {m: module_version(m) for m in ('pandas', 'numpy', 'sklearn', 'joblib', 'threadpoolctl')}}


# ---------- output guards (rerun root, symlinks/aliases, ancestors in both directions) ----------
def output_paths(out_root, cfg):
    out_root = Path(out_root)
    return {'out_root': out_root, 'report_dir': out_root / cfg['outputs']['report_dir'], 'data_dir': out_root / cfg['outputs']['data_dir']}


def _same(a, b):
    try:
        return os.path.samefile(a, b)
    except OSError:
        return False


def _inside(child, parent):
    c, p = Path(child).resolve(), Path(parent).resolve()
    if c == p or p in c.parents:
        return True
    return p.exists() and any(_same(a, p) for a in (c, *c.parents) if a.exists())


def input_rels(cfg):
    I = cfg['inputs']
    rels = [I[k]['path'] for k in ('notebook19', 'notebook20', 'notebook21')] + [I[k] for k in ('config19', 'config20', 'config21', 'manifest19', 'prepared_data_dir')]
    return rels + list(cfg['guarded_roots']) + [cfg['pins']['path']]


def protected_rels(cfg):
    return sorted(set(cfg['protected_paths']))


def guarded_paths(backend, cfg):
    backend = Path(backend)
    return [backend / r for r in sorted(set(cfg['protected_paths']) | set(input_rels(cfg)))] + [backend / cfg['notebook'], backend / 'config' / 'spatial_inference_v3.json']


def check_outputs(paths, backend, cfg):
    """Call before any mkdir/write: report/data dirs must not overlap any protected path, pinned input/root, source notebook or config (either
    direction, symlinks/aliases resolved), must not overlap each other, and must be NEW."""
    hits = []
    for label in ('report_dir', 'data_dir'):
        for p in guarded_paths(backend, cfg):
            if _inside(paths[label], p) or _inside(p, paths[label]):
                hits.append(label + ' ' + str(paths[label]) + ' overlaps protected ' + str(p))
    if _inside(paths['report_dir'], paths['data_dir']) or _inside(paths['data_dir'], paths['report_dir']):
        hits.append('report and data dirs overlap')
    if hits:
        raise ServingError('invalid_request', 'refusing output root (nothing written): ' + '; '.join(hits[:6]))
    used = [str(paths[k]) for k in ('report_dir', 'data_dir') if os.path.lexists(paths[k])]
    if used:
        raise ServingError('invalid_request', 'output already exists in ' + ', '.join(used) + '; use a NEW root, e.g. backend/rerun-spatial-serving-<id>')
    return True


def check_report_file(rel, cfg):
    if Path(rel).suffix.lower() not in cfg['report_allowed_suffixes'] or str(rel).endswith('.gz'):
        raise ServingError('invalid_request', 'refusing to write ' + str(rel) + ' into the report dir (aggregated csv/md/json only)')
    return True


def hash_paths(root, rels):
    root, out = Path(root), {}
    for rel in rels:
        p = root / rel
        if p.is_file():
            out[Path(rel).as_posix()] = sha256_file(p)
        elif p.is_dir():
            for q in sorted(p.rglob('*')):
                if q.is_file():
                    out[(Path(rel) / q.relative_to(p)).as_posix()] = sha256_file(q)
        else:
            out[Path(rel).as_posix()] = None
    return out


# ---------- trust chain ----------
def verify_file(path, expected, label):
    """Fail closed: no expected sha -> unavailable; missing -> artifact_missing; differing -> artifact_hash_mismatch."""
    path = Path(path)
    if not isinstance(expected, str) or len(expected) != 64:
        raise ServingError('artifact_unavailable', label + ': no trusted sha256 (fail closed)')
    if not path.is_file():
        raise ServingError('artifact_missing', label + ': missing ' + str(path))
    got = sha256_file(path)
    if got != expected:
        raise ServingError('artifact_hash_mismatch', label + ': sha256 ' + got[:12] + ' != trusted ' + expected[:12] + ' (' + str(path) + ')')
    return got


def pin_key(cfg, rel):
    return cfg['pins']['repo_prefix'] + Path(rel).as_posix()


def require(ok, msg):
    if not ok:
        raise ServingError('artifact_inconsistent', msg)


def selected_parts(sel):
    cid, pol = sel.split('::')
    require(pol == 'routed_context', 'serving supports the accepted routed_context policy only, got ' + sel)
    return cid, pol


def consumed_20_files(selection):
    """Relative (to the 20 data dir) files consumed by serving/parity: bundles + meta + reference requests per component, diag arrays."""
    out = {}
    for u in UNITS:
        cid, _ = selected_parts(selection[u])
        for comp in COMPONENTS:
            for suf in ('.joblib', '.meta.json', '.reference.csv'):
                out[u + ':' + comp + suf] = 'models/' + u + '/' + cid + '__' + comp + suf
        for name in (selection[u].replace('::', '__'), selection['baseline'], 'y', 'context_mask'):
            out[u + ':diag:' + name] = 'arrays/diag_' + u + '_' + name + '.npy'
    return out


def verify_chain(backend, cfg, pins):
    """Verify every pinned file, then every consumed 19/20/21 artifact against the pinned producer manifests. Returns (docs, rows).
    Order matters: nothing is executed or deserialized here; the caller loads FUNCTIONS/bundles only after this returns."""
    backend = Path(backend)
    repo, I, rows = backend.parent, cfg['inputs'], []
    if not isinstance(pins, dict) or pins.get('kind') != cfg['pins']['kind'] or not isinstance(pins.get('files'), dict):
        raise ServingError('artifact_unavailable', 'manager pins missing or of the wrong kind')
    require(pins.get('selection') == cfg['selection_expect'], 'manager pin selection ' + str(pins.get('selection')) + ' != config ' + str(cfg['selection_expect']))
    require(pins.get('experiment_root') == pin_key(cfg, I['experiment_root']) and pins.get('evaluation_root') == pin_key(cfg, I['evaluation_root']),
            'manager pin roots differ from the config roots')
    for key, h in sorted(pins['files'].items()):
        require(isinstance(key, str) and not Path(key).is_absolute() and '..' not in Path(key).parts, 'pin key must be repo-relative: ' + str(key))
        rows.append({'input': key, 'sha256': verify_file(repo / key, h, 'pinned ' + key), 'rule': 'manager pin'})
    need = [I['notebook19']['path'], I['notebook20']['path'], I['notebook21']['path'], I['config19'], I['config20'], I['config21'], I['manifest19']]
    ER, EV = Path(I['experiment_root']), Path(I['evaluation_root'])
    rep20, dat20, rep21 = ER / I['experiment_report_dir'], ER / I['experiment_data_dir'], EV / I['evaluation_report_dir']
    need += [(rep20 / n).as_posix() for n in ('spatial_experiments_manifest.json', 'experiment_lock.json', 'selection_frozen.json')]
    need += [(dat20 / n).as_posix() for n in ('arrays_index.json', 'model_bundles.json')]
    need += [(rep21 / n).as_posix() for n in ('spatial_evaluation_manifest.json', 'colour_contract_daily.json', 'colour_contract_6h.json')]
    miss = [r for r in need if pin_key(cfg, r) not in pins['files']]
    if miss:
        raise ServingError('artifact_unavailable', 'consumed inputs without a manager pin: ' + str(miss))
    rd = lambda rel: read_strict_json(backend / rel)
    docs = {'man19': rd(I['manifest19']), 'cfg19': rd(I['config19']), 'cfg20': rd(I['config20']), 'cfg21': rd(I['config21']),
            'man20': rd(rep20 / 'spatial_experiments_manifest.json'), 'lock': rd(rep20 / 'experiment_lock.json'), 'sel': rd(rep20 / 'selection_frozen.json'),
            'arrays_index': rd(dat20 / 'arrays_index.json'), 'bundles': rd(dat20 / 'model_bundles.json'), 'man21': rd(rep21 / 'spatial_evaluation_manifest.json'),
            'contracts': {u: rd(rep21 / ('colour_contract_' + u + '.json')) for u in UNITS}}
    m19, m20, m21, lock = docs['man19'], docs['man20'], docs['man21'], docs['lock']
    fs = pins.get('functions_sha256') or {}
    require(m19.get('status') == 'complete' and m19.get('functions_cells_sha256') == fs.get('notebook19'), '19 manifest not complete or FUNCTIONS sha differs from the pin')
    lock_sha = pins['files'][pin_key(cfg, (rep20 / 'experiment_lock.json').as_posix())]
    sel_sha = pins['files'][pin_key(cfg, (rep20 / 'selection_frozen.json').as_posix())]
    require(m20.get('status') == 'complete' and m20.get('experiment_lock_sha256') == lock_sha and m20.get('selection_sha256') == sel_sha,
            '20 manifest not complete or lock/selection sha differs from the pins')
    require(m20.get('report_files', {}).get('selection_frozen.json') == sel_sha and m20.get('report_files', {}).get('experiment_lock.json') == lock_sha,
            '20 manifest report hashes differ from the pinned lock/selection')
    d20 = m20.get('data_files_gitignored', {})
    for n in ('arrays_index.json', 'model_bundles.json'):
        require(d20.get(n) == pins['files'][pin_key(cfg, (dat20 / n).as_posix())], '20 manifest ' + n + ' sha differs from the pin')
    mp = lock.get('manager_pins', {})
    require(mp.get('notebook19_functions_sha256') == fs.get('notebook19'), '20 lock notebook-19 FUNCTIONS sha differs from the pin')
    require(mp.get('prepared_data_hashes') == m19.get('data_files_gitignored'), '20 lock prepared-data hashes differ from the pinned 19 manifest')
    for u in UNITS:
        s = docs['sel']['units'][u]
        require(s['selected_method'] == pins['selection'][u] and s['strongest_baseline'] == pins['selection']['baseline'] and s['recommended'] == 'candidate',
                'frozen selection ' + u + ' differs from the manager pins')
    require(docs['sel'].get('experiment_lock_sha256') == lock_sha and docs['bundles'].get('experiment_lock_sha256') == lock_sha
            and docs['arrays_index'].get('experiment_lock_sha256') == lock_sha, 'selection/bundles/arrays index do not carry the pinned lock sha')
    files20 = consumed_20_files(pins['selection'])
    for k, rel in sorted(files20.items()):
        h = d20.get(rel)
        rows.append({'input': (dat20 / rel).as_posix(), 'sha256': verify_file(backend / dat20 / rel, h, '20 artifact ' + k), 'rule': 'pinned 20 manifest'})
    for b in docs['bundles']['bundles']:
        require(b.get('sha256') == d20.get('models/' + b['unit'] + '/' + b['file']) and b.get('reload_identical') is True
                and b.get('reference_predictions_sha256') == d20.get('models/' + b['unit'] + '/' + b['file'].replace('.joblib', '.reference.csv')),
                'model_bundles.json entry ' + str(b.get('file')) + ' inconsistent with the 20 manifest')
    for u in UNITS:
        for name in (pins['selection'][u], pins['selection']['baseline'], 'y', 'context_mask'):
            a = docs['arrays_index']['units'][u]['diag'][name]
            require(a['sha256'] == d20.get('arrays/' + a['file']), 'arrays_index diag ' + u + ' ' + name + ' differs from the 20 manifest')
    require(m21.get('status') == 'complete' and m21.get('functions_sha256') == fs.get('notebook21') and m21.get('experiment_lock_sha256') == lock_sha,
            '21 manifest not complete or FUNCTIONS/lock sha differs from the pins')
    for u in UNITS:
        n = 'colour_contract_' + u + '.json'
        require(m21['report_files'].get(n) == pins['files'][pin_key(cfg, (rep21 / n).as_posix())], '21 manifest ' + n + ' differs from the pin')
        ref = docs['contracts'][u]['reference']
        require(ref.get('method') == pins['selection'][u] and ref.get('experiment_lock_sha256') == lock_sha and ref.get('selection_frozen_sha256') == sel_sha,
                'colour contract ' + u + ' reference is not the pinned selected policy')
    docs.update(lock_sha=lock_sha, sel_sha=sel_sha, files20=files20, paths={'rep20': backend / rep20, 'dat20': backend / dat20, 'rep21': backend / rep21,
                                                                             'prepared': backend / I['prepared_data_dir']})
    return docs, rows


# ---------- hash-verified FUNCTIONS of notebooks 19/20/21 ----------
def function_sources(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    cells = [(i, str(c.get('id', '')), ''.join(c['source']) if isinstance(c['source'], list) else c['source'])
             for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code']
    return cells, [s for _, _, s in cells if s.lstrip().startswith('# FUNCTIONS')]


def load_functions(nb_path, n_cells, join, expected_sha=None, code_cells_sha=None, label='notebook', exec_cells=None):
    """exec ONLY '# FUNCTIONS' cells into a fresh namespace after the cell count, the joined FUNCTIONS sha (producer convention) and,
    when given, every code-cell source sha (index:id keys) equal the trusted values. The notebook file itself is already pinned.
    exec_cells: optional subset (indices into the FUNCTIONS cells) to execute; the hash always covers ALL FUNCTIONS cells."""
    cells, funcs = function_sources(nb_path)
    sha = sha256_bytes(join.join(funcs).encode('utf-8'))
    if len(funcs) != int(n_cells) or (expected_sha is not None and sha != expected_sha):
        raise ServingError('artifact_hash_mismatch', label + ' FUNCTIONS sha ' + sha[:12] + ' (' + str(len(funcs)) + ' cells) != trusted ' + str(expected_sha)[:12])
    if code_cells_sha is not None:
        got = {str(i) + ':' + cid: sha256_bytes(src.encode('utf-8')) for i, cid, src in cells}
        if got != code_cells_sha:
            raise ServingError('artifact_hash_mismatch', label + ' code-cell hashes differ from the trusted record')
    ns = {'__name__': label.replace('-', '_') + '_functions'}
    for i, s in enumerate(funcs):
        if exec_cells is None or i in exec_cells:
            exec(compile(s, label + ':FUNCTIONS[' + str(i) + ']', 'exec'), ns)
    return ns, sha


def load_helpers(backend, cfg, pins, docs):
    """Bind the pinned 19 builder, the pinned 20 recipe/bundle loader (its H19 = the same 19 namespace) and the pinned 21 colour functions."""
    backend, I = Path(backend), cfg['inputs']
    n19, s19 = load_functions(backend / I['notebook19']['path'], I['notebook19']['functions_cells'], I['notebook19']['functions_join'],
                              pins['functions_sha256']['notebook19'], docs['lock']['manager_pins']['notebook19_code_cells_sha256'], 'notebook19')
    n20, s20 = load_functions(backend / I['notebook20']['path'], I['notebook20']['functions_cells'], I['notebook20']['functions_join'], None, None, 'notebook20')
    n20['H19'].clear()
    n20['H19'].update(n19)
    n21, s21 = load_functions(backend / I['notebook21']['path'], I['notebook21']['functions_cells'], I['notebook21']['functions_join'],
                              pins['functions_sha256']['notebook21'], None, 'notebook21', I['notebook21'].get('exec_cells'))
    need = {'19': (n19, ('load_prepared_context', 'build_feature_rows', 'validate_requests','feature_columns', 'validate_columns', 'population_axes', 'keys_from_positions',
                         'iter_population_batches', 'read_numeric_csv', 'sparse_index', 'RequestError', 'WX_DAILY', 'WX_BLOCK', 'AADT')),
            '20': (n20, ('load_bundle', 'predict_component', 'context_complete', 'routed_prediction', 'issue_time', 'training_cutoff', 'train_dates_per_block',
                         'fit_baselines', 'baseline_tables', 'baseline_predict', 'decompose', 'routed_trigger_problems')),
            '21': (n21, ('validate_colour_contract', 'assign_colour_codes', 'colour_labels', 'segment_density'))}
    for k, (ns, names) in need.items():
        miss = [n for n in names if n not in ns]
        if miss:
            raise ServingError('artifact_inconsistent', 'notebook-' + k + ' helpers missing: ' + str(miss))
    return {'19': n19, '20': n20, '21': n21, 'functions_sha256': {'notebook19': s19, 'notebook20': s20, 'notebook21': s21}}


In [ ]:
# FUNCTIONS (2/5): baseline sufficient statistics (built ONCE offline with the exact pinned notebook-20 benchmark cutoff/recipe; never inside a
# request), and load_service: trusted pins -> verified chain -> verified FUNCTIONS -> verified bundles (hash BEFORE joblib.load) -> startup
# reference self-test -> an immutable service. Any failure returns an explicit unavailable service (no hidden baseline, no fake zero).
def readonly(a):
    a = np.array(a, copy=True)
    a.setflags(write=False)
    return a


def benchmark_times(H, cfg):
    """(fit issue, training label cutoff) from the pinned 20 recipe; both must equal the declared artifact values."""
    c20, art = H['cfg20'], cfg['artifact']
    iss = H['20']['issue_time'](art['fit_issue_target_date'], c20)
    cut = H['20']['training_cutoff'](art['fit_issue_target_date'], c20)
    require(art['fit_issue_target_date'] == c20['time']['benchmark']['fit_issue_target_date'], 'artifact fit date differs from the 20 benchmark')
    require(str(iss) == art['fit_issue_local'] and str(cut) == art['training_label_cutoff_local'], 'declared artifact issue/cutoff differ from the 20 recipe')
    return iss, cut


def build_baseline_stats(H, cfg, ctx, ax, sp, unit, baseline):
    """Exact notebook-20 benchmark baseline: fit_baselines on dates < T_b (T from the benchmark cutoff) + baseline_tables over the universe."""
    n20, c20 = H['20'], H['cfg20']
    _, cut = benchmark_times(H, cfg)
    T = n20['train_dates_per_block'](ax, unit, cut)
    B = n20['fit_baselines'](ax, unit, sp, T, ctx, c20)
    tabs, ev = n20['baseline_tables'](B, ctx, ax['segments'], c20)
    tab = np.asarray(tabs[baseline], np.float64)
    require(tab.shape == (len(ax['segments']), B['nb']) and np.isfinite(tab).all() and (tab >= 0).all(), 'baseline table invalid')
    return {'unit': unit, 'baseline': baseline, 'segments': np.asarray(ax['segments'], dtype=str), 'blocks': np.asarray(ax['blocks'] if unit == '6h' else ['day'], dtype=str),
            'table': tab, 'C': np.asarray(B['C'], np.float64), 'T': np.asarray(B['T'], np.float64), 'cutoff_local': np.asarray([str(cut)], dtype=str)}, \
        dict(ev, unit=unit, training_records=B['training_records'], denominator_cells=B['denominator_cells'], train_dates_per_block=[int(t) for t in T])


def save_baseline_stats(path, stats):
    import io
    buf = io.BytesIO()
    np.savez(buf, **{k: v for k, v in stats.items() if isinstance(v, np.ndarray)})
    atomic_write_bytes(path, buf.getvalue())
    return sha256_file(path)


def load_baseline_stats(path, sha, unit, segments, n_blocks, cut):
    verify_file(path, sha, 'baseline stats ' + unit)
    with np.load(path, allow_pickle=False) as z:
        st = {k: z[k] for k in z.files}
    require(st['segments'].tolist() == list(segments), 'baseline stats segments differ from the universe')
    require(st['table'].shape == (len(segments), n_blocks) and st['table'].dtype == np.float64 and np.isfinite(st['table']).all() and (st['table'] >= 0).all(),
            'baseline stats table invalid')
    require(st['cutoff_local'].tolist() == [str(cut)], 'baseline stats cutoff differs from the benchmark cutoff')
    return readonly(st['table'])


def reference_selftest(H, ctx, unit, est, cols, ref_path, cfg20, tol):
    """Re-predict the 20 bundle reference requests (training-sample keys incl. 2021: INTERNAL component numerics, never public forecasts)."""
    ref = H['19']['read_numeric_csv'](ref_path, ['segment_id', 'target_date', 'time_block', 'lock_sha256'])
    keys = ref[['segment_id', 'target_date'] + (['time_block'] if unit == '6h' else [])]
    X, _ = H['19']['build_feature_rows'](ctx, keys, unit, 'calendar_geometry_context')
    p = H['20']['predict_component'](est, X, cols, cfg20)
    err = float(np.max(np.abs(p - ref['prediction'].to_numpy(np.float64)))) if len(ref) else math.inf
    return {'rows': int(len(ref)), 'max_abs_error': err, 'pass': bool(len(ref) > 0 and err <= tol), 'first_date': str(keys['target_date'].min()),
            'last_date': str(keys['target_date'].max())}


def unavailable_service(code, message, started):
    return types.MappingProxyType({'status': 'unavailable', 'error': types.MappingProxyType({'code': code, 'family': 'unavailable', 'message': message[:600]}),
                                   'units': types.MappingProxyType({}), 'startup_seconds': round(time.time() - started, 3), 'production_ready': False})


def load_service(backend, cfg, pins, baseline_files, selftest=True):
    """The ONLY way to build a service. backend/cfg/pins/baseline_files are server-side trusted inputs (never request data).
    baseline_files = {unit: (path, trusted sha256)}. Returns an immutable mapping; status 'available' or 'unavailable' with an error code."""
    t0 = time.time()
    try:
        require_modules(cfg['dependency_pins'])
        docs, rows = verify_chain(backend, cfg, pins)
        H = load_helpers(backend, cfg, pins, docs)
        H['cfg20'] = docs['cfg20']
        iss, cut = benchmark_times(H, cfg)
        n19, n20, n21, c20 = H['19'], H['20'], H['21'], docs['cfg20']
        ctx = n19['load_prepared_context'](docs['paths']['prepared'], docs['man19']['data_files_gitignored'])
        segs = sorted(ctx['seg'].index)
        feats = {u: {v: n19['feature_columns'](ctx['settings'], u, v) for v in ('calendar_geometry', 'calendar_geometry_context')} for u in UNITS}
        require(feats == docs['lock']['manager_pins']['feature_columns'], '19 builder feature columns differ from the 20 lock')
        units = {}
        for u in UNITS:
            sel, base = pins['selection'][u], pins['selection']['baseline']
            cid, _ = selected_parts(sel)
            trig = list(c20['candidates']['routed_trigger_columns'][u])
            probs = n20['routed_trigger_problems'](trig, u, feats, ctx['settings'], c20)
            require(not probs, 'routed trigger invalid: ' + '; '.join(probs))
            est, tests = {}, {}
            for comp, var in (('core', 'calendar_geometry'), ('context', 'calendar_geometry_context')):
                rel = docs['files20'][u + ':' + comp + '.joblib']
                path = docs['paths']['dat20'] / rel
                trusted = docs['man20']['data_files_gitignored'][rel]
                meta = read_strict_json(path.with_suffix('.meta.json'))
                require(meta.get('unit') == u and meta.get('config_id') == cid and meta.get('variant') == var and meta.get('columns') == feats[u][var]
                        and meta.get('experiment_lock_sha256') == docs['lock_sha'] and meta.get('selection_sha256') == docs['sel_sha']
                        and str(meta.get('fit_scope', '')).endswith(str(cut)), 'bundle metadata ' + rel + ' inconsistent (unit/config/variant/columns/lock/cutoff)')
                est[comp], _ = n20['load_bundle'](path, trusted)  # refuses before joblib.load unless file AND meta sha equal the trusted sha
                if selftest:
                    tests[comp] = reference_selftest(H, ctx, u, est[comp], feats[u][var], docs['paths']['dat20'] / docs['files20'][u + ':' + comp + '.reference.csv'],
                                                     c20, float(cfg['serving']['parity_abs_tolerance']))
                    require(tests[comp]['pass'], 'bundle reference self-test failed ' + u + ' ' + comp + ': ' + str(tests[comp]))
            nb = len(ctx['settings']['time_blocks']) if u == '6h' else 1
            path, sha = baseline_files[u]
            table = load_baseline_stats(path, sha, u, segs, nb, cut)
            n21['validate_colour_contract'](docs['contracts'][u], u)
            units[u] = types.MappingProxyType({
                'unit': u, 'method': sel, 'config_id': cid, 'baseline': base, 'est': types.MappingProxyType(est), 'core_cols': tuple(feats[u]['calendar_geometry']),
                'ctx_cols': tuple(feats[u]['calendar_geometry_context']), 'trigger': tuple(trig), 'contract': docs['contracts'][u], 'baseline_table': table,
                'blocks': tuple(ctx['settings']['time_blocks']) if u == '6h' else ('day',), 'selftest': types.MappingProxyType(tests),
                'bundle_sha256': types.MappingProxyType({c: docs['man20']['data_files_gitignored'][docs['files20'][u + ':' + c + '.joblib']] for c in COMPONENTS}),
                'baseline_stats_sha256': sha})
        seg_index = pd.Index(segs)
        km = readonly(ctx['seg'].loc[segs, 'geom_length_m'].to_numpy(np.float64) / 1000.0)
        require(np.isfinite(km).all() and (km > 0).all(), 'segment lengths must be finite and positive')
        art = cfg['artifact']
        meta = {'model_version': art['model_version'], 'artifact_kind': art['kind'], 'default_for_api': art['default_for_api'], 'fit_issue_local': str(iss),
                'training_label_cutoff_local': str(cut), 'supported_target_start': art['supported_target_start'], 'supported_target_end': art['supported_target_end'],
                'observed_diagnostic_end': art['observed_diagnostic_end'], 'extended_refit': art['extended_refit'], 'oof_fold_models': art['oof_fold_models'],
                'experiment_lock_sha256': docs['lock_sha'], 'selection_sha256': docs['sel_sha'], 'functions_sha256': H['functions_sha256'],
                'verified_inputs': len(rows), 'supported_segments': len(segs), 'label_meaning': cfg['label_meaning'], 'provenance': cfg['provenance'],
                'caveats': list(cfg['caveats']), 'max_batch': int(cfg['serving']['max_batch']), 'threads': int(cfg['serving']['threads'])}
        return types.MappingProxyType({'status': 'available', 'error': None, 'units': types.MappingProxyType(units), 'ctx': ctx, 'H': H, 'cfg20': c20,
                                       'seg_index': seg_index, 'seg_km': km, 'issue': iss, 'cutoff': cut, 'meta': types.MappingProxyType(meta),
                                       'settings': ctx['settings'], 'cfg': cfg, 'verified_rows': tuple(rows), 'startup_seconds': round(time.time() - t0, 3),
                                       'production_ready': False})
    except ServingError as e:
        return unavailable_service(e.code, e.message, t0)
    except Exception as e:  # e.g. a corrupt bundle that still matched nothing trusted: explicit unavailable, never a fallback
        return unavailable_service('artifact_unavailable', type(e).__name__ + ': ' + str(e), t0)


def health(service):
    """Startup availability is reported separately from readiness: production_ready is always false for this research artifact."""
    ok = service['status'] == 'available'
    out = {'startup_status': service['status'], 'available_units': sorted(service['units']) if ok else [], 'production_ready': False,
           'research_ready': ok, 'startup_seconds': service['startup_seconds'], 'error': dict(service['error']) if service['error'] else None}
    if ok:
        out['model_version'] = service['meta']['model_version']
        out['selftest'] = {u: {c: dict(v) for c, v in s['selftest'].items()} for u, s in service['units'].items()}
    return jsonable(out)


In [ ]:
# FUNCTIONS (3/5): the public request contract and responses. predict_batch (selected policy) and predict_baseline_batch (locked strong baseline)
# share the same keys, guards and as-of rules. Features ONLY via the pinned 19 build_feature_rows; components/routing/baseline lookup ONLY via
# pinned 20 FUNCTIONS; colours ONLY via pinned 21 FUNCTIONS. Pure: no fit, no download, no write, no path from requests.
COMP_CODES = ('core', 'context', 'baseline')


def need_fields(unit):
    return ['segment_id', 'target_date'] + (['time_block'] if unit == '6h' else [])


def _require_available(service):
    if not isinstance(service, types.MappingProxyType) or service.get('status') != 'available':
        err = service.get('error') if isinstance(service, types.MappingProxyType) else None
        raise ServingError('artifact_unavailable', 'service unavailable' + (' (' + err['code'] + ': ' + err['message'][:200] + ')' if err else ''))


def _flag_rows(code, message, bad):
    bad = np.asarray(bad, bool)
    if bad.any():
        raise ServingError(code, message, np.flatnonzero(bad))


def precheck_like_19(ctx, R, unit):
    """The pinned 19 validate_requests checks in the SAME family order, raised as ServingError with explicit row positions. Needed because the
    pinned 19 RequestError stores `list(rows or [])`: 2+ positions raise an ambiguous-truth ValueError (would surface as artifact_unavailable) and a
    lone row 0 is dropped. The pinned 19 validator still runs afterwards as the authority; feature logic is untouched."""
    s, d = ctx['settings'], R['target_date']
    iso = d.str.fullmatch(r'\d{4}-\d{2}-\d{2}').fillna(False).to_numpy(bool) & pd.to_datetime(d, format='%Y-%m-%d', errors='coerce').notna().to_numpy()
    _flag_rows('invalid_date', 'target_date must be an ISO local date', ~iso)
    _flag_rows('date_out_of_range', 'supported research dates ' + s['research_start'] + '..' + s['research_end'],
               ((d < s['research_start']) | (d > s['research_end'])).to_numpy())
    if unit == '6h':
        _flag_rows('invalid_time_block', 'time_block must be one of ' + str(s['time_blocks']), ~R['time_block'].isin(s['time_blocks']).to_numpy())
    _flag_rows('duplicate_keys', 'request keys must be unique', R.duplicated(list(R.columns)).to_numpy())
    unknown = ~R['segment_id'].isin(ctx['seg'].index).to_numpy()
    if unknown.any():
        uns = R['segment_id'].isin(ctx['unsupported']).to_numpy()
        _flag_rows('unsupported_segment' if (uns & unknown).any() else 'unknown_segment',
                   'segments outside the qualified DOH 2000 m universe have no prediction (grey/unsupported, never zero)', unknown)


def validate_public(service, unit, requests):
    """Strict public contract (before any feature work): unit exactly 'daily'|'6h'; list of 1..max_batch objects with EXACTLY the unit's fields; every
    value a non-empty trimmed str (bool/number/None refused, no coercion); then the pinned 19 validator (ISO dates, blocks, duplicates, universe) and the
    artifact as-of guard: issue(target) >= declared fit issue AND training label cutoff <= issue - label latency; dates outside the artifact range refused."""
    _require_available(service)
    if not isinstance(unit, str) or isinstance(unit, bool) or unit not in UNITS:
        raise ServingError('invalid_unit', 'unit must be exactly "daily" or "6h"')
    if not isinstance(requests, (list, tuple)):
        raise ServingError('invalid_request', 'requests must be a JSON array of objects')
    mx = int(service['meta']['max_batch'])
    if not 1 <= len(requests) <= mx:
        raise ServingError('batch_size', 'rows must be 1..' + str(mx))
    need = need_fields(unit)
    # each family reports EVERY offending row (whole batch refused atomically; first family wins, fixed order)
    _flag_rows('invalid_request', 'each request must be an object', [not isinstance(r, dict) for r in requests])
    _flag_rows('invalid_fields', 'fields must be exactly ' + str(need) + (' (time_block only for 6h)' if unit == 'daily' else ''),
               [set(r) != set(need) for r in requests])
    _flag_rows('invalid_value', '/'.join(need) + ' must be non-empty trimmed strings (no bool/number coercion)',
               [any(not isinstance(r[k], str) or r[k] != r[k].strip() or r[k] == '' for k in need) for r in requests])
    R = pd.DataFrame([{k: r[k] for k in need} for r in requests], columns=need)
    H, art = service['H'], service['meta']
    precheck_like_19(service['ctx'], R, unit)
    try:
        R = H['19']['validate_requests'](service['ctx'], R, unit)
    except H['19']['RequestError'] as e:
        raise ServingError(e.code if e.code in INVALID_CODES + COVERAGE_CODES else 'invalid_request', str(e).split(': ', 1)[-1], e.rows)
    early = R['target_date'] < art['supported_target_start']
    if early.any():
        raise ServingError('date_before_artifact_asof', 'the ' + art['artifact_kind'] + ' artifact was fitted as of ' + art['fit_issue_local'] +
                           ' (labels to ' + art['training_label_cutoff_local'] + '); earlier targets would be a false prospective forecast. Supported ' +
                           art['supported_target_start'] + '..' + art['supported_target_end'], np.flatnonzero(early.to_numpy()))
    late = R['target_date'] > art['supported_target_end']
    if late.any():
        raise ServingError('date_out_of_range', 'supported target dates ' + art['supported_target_start'] + '..' + art['supported_target_end'], np.flatnonzero(late.to_numpy()))
    c20 = service['cfg20']
    iss = pd.to_datetime(R['target_date'], format='%Y-%m-%d').map(lambda t: H['20']['issue_time'](t, c20))
    lat = pd.Timedelta(days=int(c20['time']['label_latency_days']))
    bad = (iss < service['issue']) | (iss - lat < service['cutoff'])
    if bad.any():
        raise ServingError('date_before_artifact_asof', 'issue/label-latency rule violated', np.flatnonzero(bad.to_numpy()))
    return R


def predict_arrays(service, unit, R, which):
    """Internal numeric path shared by the public batch API (<= max_batch, after validate_public) and the offline full-population parity
    (<= parity_batch_rows, pinned 19 validator only). which: 'selected' | 'baseline'. Returns dict of aligned arrays + the 19 X/M frames."""
    S, H = service['units'][unit], service['H']
    n19, n20, c20 = H['19'], H['20'], service['cfg20']
    if len(R) > int(service['cfg']['serving']['parity_batch_rows']):
        raise ServingError('batch_size', 'internal batches are bounded to ' + str(service['cfg']['serving']['parity_batch_rows']) + ' rows')
    X, M = n19['build_feature_rows'](service['ctx'], R, unit, 'calendar_geometry_context')
    if list(X.columns) != list(S['ctx_cols']) or list(X.columns[:len(S['core_cols'])]) != list(S['core_cols']):
        raise ServingError('artifact_inconsistent', 'feature columns/order differ from the pinned bundle columns')
    si = service['seg_index'].get_indexer(R['segment_id'])
    if (si < 0).any():
        raise ServingError('unknown_segment', 'segment outside the supported universe', np.flatnonzero(si < 0))
    bi = pd.Index(S['blocks']).get_indexer(R['time_block']) if unit == '6h' else np.zeros(len(R), np.int64)
    mask = np.asarray(n20['context_complete'](X, unit, c20), bool)
    try:
        if which == 'selected':
            pc = n20['predict_component'](S['est']['core'], X, list(S['core_cols']), c20)
            px = n20['predict_component'](S['est']['context'], X, list(S['ctx_cols']), c20)
            lam = np.asarray(n20['routed_prediction'](pc, px, mask), np.float64)
            comp = np.where(mask, 1, 0).astype(np.int8)
        elif which == 'baseline':
            lam = np.asarray(n20['baseline_predict']({S['baseline']: S['baseline_table']}, si, bi, None, None, [S['baseline']])[S['baseline']], np.float64)
            comp = np.full(len(R), 2, np.int8)
        else:
            raise ServingError('invalid_request', 'which must be selected|baseline')
    except ServingError:
        raise
    except Exception as e:  # the 20 guard refuses non-finite/negative/misshaped component output: never replaced by a fallback
        raise ServingError('prediction_invalid', type(e).__name__ + ': ' + str(e)[:300])
    if lam.shape != (len(R),) or not np.isfinite(lam).all() or (lam < 0).any():
        raise ServingError('prediction_invalid', 'prediction non-finite, negative or misshaped')
    return {'lam': lam, 'comp': comp, 'mask': mask, 'si': si, 'bi': bi, 'X': X, 'M': M}


def _txt(v):
    return None if v is None or (isinstance(v, float) and not math.isfinite(v)) else str(v)


def build_rows(service, unit, R, A, which):
    S, H, meta = service['units'][unit], service['H'], service['meta']
    km = service['seg_km'][A['si']]
    dens = H['21']['segment_density'](A['lam'], km * 1000.0)[0]
    if which == 'selected':
        labels = H['21']['colour_labels'](H['21']['assign_colour_codes'](A['lam'], km * 1000.0, S['contract'], unit))
    else:
        labels = np.full(len(R), None, dtype=object)
    X, M, trig = A['X'], A['M'], list(S['trigger'])
    fin = np.isfinite(X[trig].to_numpy(np.float64))
    bank = X['bank_holiday_asof'].to_numpy(np.float64)
    method = S['method'] if which == 'selected' else S['baseline']
    out = []
    for i in range(len(R)):
        d, lab = R['target_date'].iat[i], labels[i]
        miss = [c for c, ok in zip(trig, fin[i]) if not ok]
        src = {'weather_daily_rain': _txt(M['wx_rain_reason'].iat[i]), 'weather_daily_temp': _txt(M['wx_temp_reason'].iat[i]), 'aadt': _txt(M['aadt_reason'].iat[i]),
               'bank_calendar': 'known' if math.isfinite(bank[i]) else 'unknown_preserved_missing', 'bank_calendar_coverage': _txt(M['bank_calendar_coverage'].iat[i])}
        if unit == '6h':
            src.update(weather_block_rain=_txt(M['wx_block_rain_reason'].iat[i]), weather_block_temp=_txt(M['wx_block_temp_reason'].iat[i]))
        research = d > meta['observed_diagnostic_end']
        out.append({
            'segment_id': R['segment_id'].iat[i], 'target_date': d, 'time_block': R['time_block'].iat[i] if unit == '6h' else None, 'unit': unit,
            'horizon': (('local day ' + d) if unit == 'daily' else ('6 h block ' + R['time_block'].iat[i] + ' of ' + d)) + ' (Asia/Bangkok)',
            'issue_time_local': _txt(M['issue_time_local'].iat[i]), 'issue_time_utc': _txt(M['issue_time_utc'].iat[i]),
            'model_version': meta['model_version'], 'method': method, 'role': 'selected_policy' if which == 'selected' else 'strong_baseline_comparison',
            'component_used': COMP_CODES[int(A['comp'][i])],
            'routing_reason': ('baseline_has_no_routing' if which == 'baseline' else
                               ('context_complete' if not miss else 'core_fallback_missing_context:' + ','.join(miss))),
            'expected_recorded_count': float(A['lam'][i]), 'segment_km': float(km[i]), 'density_per_km': float(dens[i]),
            'colour': lab, 'colour_hex': S['contract']['hex'][lab] if lab else None, 'colour_text': S['contract']['accessible_text'][lab] if lab else None,
            'colour_status': 'relative_contract_' + S['contract']['version'] if lab else 'not_applicable_for_baseline',
            'research_scenario': bool(research),
            'data_period': 'research_scenario_no_observed_ground_truth' if research else 'previously_seen_diagnostic_2024_2025',
            'coverage': 'supported_qualified_doh_segment', 'source_availability': src})
    return out


def error_response(e, unit=None):
    return {'ok': False, 'unit': unit if isinstance(unit, str) else None, 'production_ready': False,
            'error': {'code': e.code, 'family': error_family(e.code), 'message': e.message[:600], 'rows': list(e.rows)}}


def _respond(service, unit, requests, which):
    try:
        R = validate_public(service, unit, requests)
        A = predict_arrays(service, unit, R, which)
        res = {'ok': True, 'unit': unit, 'rows': len(R), 'model_version': service['meta']['model_version'],
               'method': service['units'][unit]['method'] if which == 'selected' else service['units'][unit]['baseline'],
               'quantity': 'expected SELECTED+LINKED recorded count per segment and horizon (lambda); not a probability',
               'production_ready': False, 'results': build_rows(service, unit, R, A, which), 'caveats': list(service['meta']['caveats'])}
    except ServingError as e:
        res = error_response(e, unit)
    except Exception as e:  # unexpected internal failure: explicit unavailable, never a partial or zero answer
        res = error_response(ServingError('artifact_unavailable', type(e).__name__ + ': ' + str(e)[:300]), unit)
    res = jsonable(res)
    strict_dumps(res)  # refuses NaN/Infinity: every response is finite JSON
    return res


def predict_batch(service, unit, requests):
    """Public selected-policy prediction; output order == request order."""
    return _respond(service, unit, requests, 'selected')


def predict_baseline_batch(service, unit, requests):
    """Public locked strong-baseline prediction (2026 comparison), same keys/guards/as-of rules as predict_batch."""
    return _respond(service, unit, requests, 'baseline')


def metadata(service):
    if service['status'] != 'available':
        return jsonable({'ok': False, 'production_ready': False, 'error': dict(service['error'])})
    m = dict(service['meta'])
    m['units'] = {u: {'method': s['method'], 'baseline': s['baseline'], 'fields': need_fields(u), 'time_blocks': list(s['blocks']) if u == '6h' else None,
                      'routed_trigger_columns': list(s['trigger']), 'colour_cutoffs': s['contract']['raw_cutoffs'], 'colour_density_unit': s['contract']['density_unit'],
                      'bundle_sha256': dict(s['bundle_sha256']), 'baseline_stats_sha256': s['baseline_stats_sha256']} for u, s in service['units'].items()}
    m.update(ok=True, production_ready=False, error_codes={'invalid': list(INVALID_CODES), 'coverage': list(COVERAGE_CODES), 'unavailable': list(UNAVAILABLE_CODES)})
    return jsonable(m)


def coverage(service, segment_ids):
    """Per id: supported (qualified DOH universe) | unsupported_segment (known non-qualified road) | unknown_segment. No snapping."""
    try:
        _require_available(service)
        if not isinstance(segment_ids, (list, tuple)) or not 1 <= len(segment_ids) <= int(service['meta']['max_batch']):
            raise ServingError('batch_size', 'segment_ids must be a list of 1..' + str(service['meta']['max_batch']))
        if any(not isinstance(s, str) or s != s.strip() or not s for s in segment_ids):
            raise ServingError('invalid_value', 'segment ids must be non-empty trimmed strings')
        uns = service['ctx']['unsupported']
        st = ['supported' if s in service['seg_index'] else ('unsupported_segment' if s in uns else 'unknown_segment') for s in segment_ids]
        return jsonable({'ok': True, 'production_ready': False, 'results': [{'segment_id': s, 'coverage': c} for s, c in zip(segment_ids, st)]})
    except ServingError as e:
        return jsonable(error_response(e))


In [ ]:
# FUNCTIONS (4/5): offline verification of the runtime. Full saved 2024/25 diagnostic populations streamed in <= parity_batch_rows batches through
# the SAME predict_arrays path and compared with the hash-verified notebook-20 arrays (selected policy, strong baseline, routing mask); traced
# deterministic reference requests through the PUBLIC API; robustness/tamper cases; CSV reload parity; measured latency (no SLA).
def open_saved(service_docs, unit, name):
    a = service_docs['arrays_index']['units'][unit]['diag'][name]
    p = service_docs['paths']['dat20'] / 'arrays' / a['file']
    verify_file(p, service_docs['man20']['data_files_gitignored']['arrays/' + a['file']], 'saved diag ' + unit + ' ' + name)
    arr = np.load(p, mmap_mode='r', allow_pickle=False)
    require(list(arr.shape) == list(a['shape']) and str(arr.dtype) == a['dtype'], 'saved diag array shape/dtype differs from arrays_index')
    return arr


def diag_window(service_docs, unit):
    P = service_docs['arrays_index']['units'][unit]['diag_periods']
    return P[0]['start'], P[-1]['end'], int(sum(p['rows'] for p in P))


def routing_scope(context_rows, rows):
    """What the SAVED 2024/25 mask actually covers (reported, never assumed): no observed context -> core-only scope."""
    return 'core_only_no_observed_context' if context_rows == 0 else ('context_only' if context_rows == rows else 'mixed_context_and_core')


def full_parity(service, docs, ax, unit, out_dir):
    """Stream EVERY saved 2024/25 cell. Writes runtime selected/baseline/mask arrays (float64/uint8 .npy) for the independent verifier."""
    S, H = service['units'][unit], service['H']
    sel, base = S['method'], S['baseline']
    saved = {k: open_saved(docs, unit, k) for k in (sel, base, 'context_mask')}
    start, end, n = diag_window(docs, unit)
    nb = len(ax['blocks']) if unit == '6h' else 1
    per_day = len(ax['segments']) * nb
    d0 = ax['date_pos'][start]
    require(n == (ax['date_pos'][end] - d0 + 1) * per_day and all(len(v) == n for v in saved.values()), 'saved diag population size mismatch')
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    files = {'selected': out_dir / ('runtime_diag_' + unit + '_selected.npy'), 'baseline': out_dir / ('runtime_diag_' + unit + '_baseline.npy'),
             'context_mask': out_dir / ('runtime_diag_' + unit + '_context_mask.npy')}
    mm = {k: np.lib.format.open_memmap(p, mode='w+', dtype=np.uint8 if k == 'context_mask' else np.float64, shape=(n,)) for k, p in files.items()}
    mm['selected'][:] = np.nan
    mm['baseline'][:] = np.nan
    mm['context_mask'][:] = 255
    err = {'selected': 0.0, 'baseline': 0.0}
    over, mask_bad, ctx_rows, saved_ctx, written, t0 = {'selected': 0, 'baseline': 0}, 0, 0, 0, 0, time.time()
    tol = float(service['cfg']['serving']['parity_abs_tolerance'])
    for a, keys in H['19']['iter_population_batches'](ax, unit, int(service['cfg']['serving']['parity_batch_rows']), start, end):
        loc = np.arange(a, a + len(keys), dtype=np.int64) - d0 * per_day
        A = predict_arrays(service, unit, keys, 'selected')
        lb = np.asarray(H['20']['baseline_predict']({base: S['baseline_table']}, A['si'], A['bi'], None, None, [base])[base], np.float64)
        for k, v in (('selected', A['lam']), ('baseline', lb)):
            ref = np.asarray(saved[sel if k == 'selected' else base][loc[0]:loc[-1] + 1], np.float64)
            d = np.abs(v - ref)
            err[k] = max(err[k], float(d.max()))
            over[k] += int((d > tol).sum())
            mm[k][loc[0]:loc[-1] + 1] = v
        sm = np.asarray(saved['context_mask'][loc[0]:loc[-1] + 1])
        mask_bad += int((sm != A['mask'].astype(np.uint8)).sum())
        ctx_rows += int(A['mask'].sum())
        saved_ctx += int((sm == 1).sum())
        mm['context_mask'][loc[0]:loc[-1] + 1] = A['mask'].astype(np.uint8)
        written += len(keys)
    meta = {}
    for k, m in mm.items():
        m.flush()
        del m
        a_ = np.load(files[k], mmap_mode='r', allow_pickle=False)
        ok = bool(np.isin(a_, (0, 1)).all()) if k == 'context_mask' else bool(np.isfinite(a_).all() and (a_ >= 0).all())
        require(ok, 'runtime array ' + k + ' incomplete or invalid')
        meta[k] = {'file': files[k].name, 'sha256': sha256_file(files[k]), 'rows': n}
    mm.clear()
    rows = [{'unit': unit, 'scope': 'seen_2024_2025_full_population', 'method': m, 'role': r, 'rows': written, 'population_rows': n,
             'max_abs_error': err[k], 'cells_over_tolerance': over[k], 'tolerance': tol, 'routing_mask_mismatches': mask_bad, 'context_rows': ctx_rows,
             'core_rows': written - ctx_rows, 'saved_mask_context_rows': saved_ctx, 'observed_routing_scope': routing_scope(saved_ctx, written), 'batch_rows': int(service['cfg']['serving']['parity_batch_rows']), 'seconds': round(time.time() - t0, 1),
             'saved_array_sha256': docs['man20']['data_files_gitignored']['arrays/' + docs['arrays_index']['units'][unit]['diag'][m]['file']],
             'runtime_array_sha256': meta[k]['sha256']} for k, m, r in (('selected', sel, 'selected_policy'), ('baseline', base, 'strong_baseline'))]
    return rows, meta


def selftest_rows(service):
    return [dict(unit=u, component=c, scope='bundle_reference_requests_internal_component_check (training-sample keys incl. 2021; NOT prospective)', **dict(v))
            for u, s in service['units'].items() for c, v in s['selftest'].items()]


def reference_requests(service, ax, docs, unit, cfg):
    """Deterministic public requests: (1-future_share) drawn from the saved 2024/25 population (each traced to its array offset), the rest from
    2026-27 research dates (no ground truth, no saved array). Positions are unique; order is shuffled deterministically."""
    rr = cfg['serving']['reference_requests']
    rng = np.random.default_rng(int(rr['seed']) + (0 if unit == 'daily' else 1))
    n = int(rr['rows_per_unit'])
    nf = int(round(n * float(rr['future_share'])))
    start, end, N = diag_window(docs, unit)
    nb = len(ax['blocks']) if unit == '6h' else 1
    off = np.sort(rng.choice(N, n - nf, replace=False))
    d0 = ax['date_pos'][start]
    seen_keys = service['H']['19']['keys_from_positions'](ax, unit, off + d0 * len(ax['segments']) * nb)
    fut_dates = pd.date_range('2026-01-01', service['meta']['supported_target_end'], freq='D').strftime('%Y-%m-%d').to_numpy(object)
    segs = np.asarray(ax['segments'], dtype=object)
    fk = pd.DataFrame({'segment_id': segs[rng.integers(0, len(segs), nf * 3)], 'target_date': fut_dates[rng.integers(0, len(fut_dates), nf * 3)]})
    if unit == '6h':
        fk['time_block'] = np.asarray(ax['blocks'], dtype=object)[rng.integers(0, nb, nf * 3)]
    fk = fk.drop_duplicates().head(nf)
    require(len(fk) == nf, 'could not draw unique future requests')
    allk = pd.concat([seen_keys.assign(diag_offset=off), fk.assign(diag_offset=-1)], ignore_index=True)
    allk = allk.iloc[rng.permutation(len(allk))].reset_index(drop=True)
    return allk


def run_reference(service, docs, unit, refk):
    """Public predict_batch / predict_baseline_batch in max_batch chunks; seen rows compared with the saved 20 arrays at their offsets."""
    S, mx = service['units'][unit], int(service['meta']['max_batch'])
    saved = {'selected': open_saved(docs, unit, S['method']), 'baseline': open_saved(docs, unit, S['baseline'])}
    smask = open_saved(docs, unit, 'context_mask')
    need = need_fields(unit)
    rows, responses = [], []
    for a in range(0, len(refk), mx):
        chunk = refk.iloc[a:a + mx]
        reqs = [{k: str(r[k]) for k in need} for _, r in chunk.iterrows()]
        rs = {'selected': predict_batch(service, unit, reqs), 'baseline': predict_baseline_batch(service, unit, reqs)}
        for which, res in rs.items():
            require(res['ok'], 'reference request failed: ' + str(res.get('error')))
            responses.append({'which': which, 'requests': reqs, 'response': res})
            for (_, r), out in zip(chunk.iterrows(), res['results']):
                require(out['segment_id'] == r['segment_id'] and out['target_date'] == r['target_date'], 'response order differs from request order')
                o = int(r['diag_offset'])
                exp = float(saved[which][o]) if o >= 0 else None
                rows.append({'unit': unit, 'which': which, 'segment_id': out['segment_id'], 'target_date': out['target_date'], 'time_block': out['time_block'],
                             'diag_offset': o, 'saved_context_mask': int(smask[o]) if o >= 0 else -1,
                             'expected_recorded_count': out['expected_recorded_count'], 'saved_20_value': exp,
                             'abs_error': abs(out['expected_recorded_count'] - exp) if exp is not None else None, 'component_used': out['component_used'],
                             'routing_reason': out['routing_reason'], 'colour': out['colour'], 'density_per_km': out['density_per_km'],
                             'research_scenario': out['research_scenario'], 'bank_calendar': out['source_availability']['bank_calendar']})
    return pd.DataFrame(rows), responses


def _with_unit(service, unit, **changes):
    """A modified COPY of a service (robustness only; the original stays untouched)."""
    units = dict(service['units'])
    units[unit] = types.MappingProxyType(dict(units[unit], **changes))
    return types.MappingProxyType(dict(service, units=types.MappingProxyType(units)))


class _ConstEstimator:
    def __init__(self, v):
        self.v = v

    def predict(self, X):
        return np.full(len(X), self.v)


def robustness_cases(service, unit):
    """(case, unit, callable -> response, expected error code or None). Every response must also be strict finite JSON."""
    ctx = service['ctx']
    seg = service['seg_index'][0]
    blk = service['units']['6h']['blocks'][0]
    ok = {'segment_id': seg, 'target_date': '2024-03-01', **({'time_block': blk} if unit == '6h' else {})}
    uns = sorted(ctx['unsupported'])
    C = [('valid_row', lambda: predict_batch(service, unit, [ok]), None),
         ('wrong_unit_text', lambda: predict_batch(service, 'weekly', [ok]), 'invalid_unit'),
         ('unit_bool', lambda: predict_batch(service, True, [ok]), 'invalid_unit'),
         ('not_a_list', lambda: predict_batch(service, unit, ok), 'invalid_request'),
         ('empty_batch', lambda: predict_batch(service, unit, []), 'batch_size'),
         ('oversize_batch_129', lambda: predict_batch(service, unit, [dict(ok, target_date=d) for d in
                                                                       pd.date_range('2024-01-01', periods=129).strftime('%Y-%m-%d')]), 'batch_size'),
         ('extra_field', lambda: predict_batch(service, unit, [dict(ok, path='/etc/passwd')]), 'invalid_fields'),
         ('bool_value', lambda: predict_batch(service, unit, [dict(ok, segment_id=True)]), 'invalid_value'),
         ('numeric_date', lambda: predict_batch(service, unit, [dict(ok, target_date=20240301)]), 'invalid_value'),
         ('padded_value', lambda: predict_batch(service, unit, [dict(ok, segment_id=' ' + seg)]), 'invalid_value'),
         ('non_iso_date', lambda: predict_batch(service, unit, [dict(ok, target_date='2024-3-01')]), 'invalid_date'),
         ('impossible_date', lambda: predict_batch(service, unit, [dict(ok, target_date='2024-02-30')]), 'invalid_date'),
         ('duplicate_keys', lambda: predict_batch(service, unit, [ok, dict(ok)]), 'duplicate_keys'),
         ('date_before_asof_2023', lambda: predict_batch(service, unit, [dict(ok, target_date='2023-12-31')]), 'date_before_artifact_asof'),
         ('date_2021_internal_reference_not_public', lambda: predict_batch(service, unit, [dict(ok, target_date='2021-06-01')]), 'date_before_artifact_asof'),
         ('date_after_2027', lambda: predict_batch(service, unit, [dict(ok, target_date='2028-01-01')]), 'date_out_of_range'),
         ('unknown_segment', lambda: predict_batch(service, unit, [dict(ok, segment_id='not-a-doh-segment')]), 'unknown_segment'),
         ('baseline_unknown_segment', lambda: predict_baseline_batch(service, unit, [dict(ok, segment_id='not-a-doh-segment')]), 'unknown_segment'),
         ('baseline_before_asof', lambda: predict_baseline_batch(service, unit, [dict(ok, target_date='2023-06-01')]), 'date_before_artifact_asof'),
         ('feature_order_tampered', lambda: predict_batch(_with_unit(service, unit, ctx_cols=tuple(reversed(service['units'][unit]['ctx_cols']))), unit, [ok]),
          'artifact_inconsistent'),
         ('negative_prediction', lambda: predict_batch(_with_unit(service, unit, est=types.MappingProxyType({'core': _ConstEstimator(-1.0), 'context': _ConstEstimator(-1.0)})),
                                                         unit, [ok]), 'prediction_invalid'),
         ('nonfinite_prediction', lambda: predict_batch(_with_unit(service, unit, est=types.MappingProxyType({'core': _ConstEstimator(np.nan), 'context': _ConstEstimator(np.inf)})),
                                                          unit, [ok]), 'prediction_invalid'),
         ('unavailable_service', lambda: predict_batch(unavailable_service('artifact_hash_mismatch', 'fixture', time.time()), unit, [ok]), 'artifact_unavailable')]
    if unit == '6h':
        C += [('block_missing', lambda: predict_batch(service, unit, [{'segment_id': seg, 'target_date': '2024-03-01'}]), 'invalid_fields'),
              ('block_invalid', lambda: predict_batch(service, unit, [dict(ok, time_block='06-13')]), 'invalid_time_block')]
    else:
        C += [('block_extra_for_daily', lambda: predict_batch(service, unit, [dict(ok, time_block=blk)]), 'invalid_fields')]
    if uns:
        C.append(('unsupported_geometry', lambda: predict_batch(service, unit, [dict(ok, segment_id=uns[0])]), 'unsupported_segment'))
    return C + multi_row_cases(service, unit)


def multi_row_cases(service, unit):
    """Multi-row invalid/coverage batches: (case, callable, expected code, expected offending rows). Each batch mixes good and bad rows (incl. row 0
    and 2+ bad rows); the whole batch must be refused with the stable family and exact row positions, never a partial answer."""
    segs = [str(s) for s in service['seg_index'][:4]]
    blk = service['units']['6h']['blocks']
    q = lambda i, d='2024-03-01', **kw: dict({'segment_id': segs[i], 'target_date': d, **({'time_block': blk[0]} if unit == '6h' else {})}, **kw)
    C = []
    for which, fn in (('selected', predict_batch), ('baseline', predict_baseline_batch)):
        C += [('multi_' + which + '_before_asof_rows_0_2', (lambda fn=fn: fn(service, unit, [q(0, '2023-12-31'), q(1), q(2, '2023-06-01'), q(3)])),
               'date_before_artifact_asof', [0, 2]),
              ('multi_' + which + '_before_asof_row_0_only', (lambda fn=fn: fn(service, unit, [q(0, '2022-05-05'), q(1)])), 'date_before_artifact_asof', [0]),
              ('multi_' + which + '_after_2027_rows_1_3', (lambda fn=fn: fn(service, unit, [q(0), q(1, '2028-01-01'), q(2), q(3, '2030-06-01')])),
               'date_out_of_range', [1, 3]),
              ('multi_' + which + '_unknown_segment_rows_0_1', (lambda fn=fn: fn(service, unit, [q(0, segment_id='nope-a'), q(1, segment_id='nope-b'), q(2)])),
               'unknown_segment', [0, 1]),
              ('multi_' + which + '_invalid_date_rows_0_2', (lambda fn=fn: fn(service, unit, [q(0, '2024-02-30'), q(1), q(2, '2024-3-01')])), 'invalid_date', [0, 2]),
              ('multi_' + which + '_invalid_value_rows_0_1', (lambda fn=fn: fn(service, unit, [q(0, segment_id=' x'), q(1, target_date=20240301), q(2)])),
               'invalid_value', [0, 1]),
              ('multi_' + which + '_duplicate_rows_2_3', (lambda fn=fn: fn(service, unit, [q(0), q(1), q(0), q(1)])), 'duplicate_keys', [2, 3]),
              ('multi_' + which + '_invalid_date_precedes_coverage', (lambda fn=fn: fn(service, unit, [q(0, segment_id='nope'), q(1, '2024-13-01'), q(2)])),
               'invalid_date', [1])]
        if unit == '6h':
            C.append(('multi_' + which + '_empty_block_value_precedes_bad_block',(lambda fn=fn: fn(service, unit, [q(0, time_block='24-30'), q(1), q(2, time_block='')])),
                      'invalid_value', [2]))
            C.append(('multi_' + which + '_bad_block_rows_0_1', (lambda fn=fn: fn(service, unit, [q(0, time_block='24-30'), q(1, time_block='06-13'), q(2)])),
                      'invalid_time_block', [0, 1]))
    return C


def run_cases(cases):
    """cases: (name, callable, expected code or None[, expected offending rows]). An error response must carry no results (atomic refusal)."""
    rows = []
    for name, fn, want, *rest in cases:
        want_rows = rest[0] if rest else None
        r = fn()
        strict_dumps(r)
        got = None if r.get('ok') else r['error']['code']
        got_rows = None if r.get('ok') else list(r['error']['rows'])
        passed = got == want and (want is None or 'results' not in r) and (want_rows is None or got_rows == list(want_rows))
        rows.append({'case': name, 'expected_code': want, 'got_code': got, 'family': error_family(got) if got else None,
                     'expected_rows': None if want_rows is None else json.dumps(list(want_rows)), 'got_rows': None if got_rows is None else json.dumps(got_rows),
                     'passed': bool(passed), 'message': (r.get('error') or {}).get('message', '')[:200]})
    return rows


SYNTHETIC_EVIDENCE = 'SYNTHETIC_FIXTURE_runtime_routing_check_NOT_observed_2024_2025_evidence'
SYNTHETIC_KINDS = ('context_complete', 'missing_first_trigger', 'all_missing', 'context_complete', 'missing_last_trigger', 'all_missing')


def synthetic_routing_probe(service, unit, fill_value=1.0):
    """CLEARLY SYNTHETIC routing fixture (the real 2024/25 diagnostic mask is core-only: no observed weather/AADT context). The pinned 19
    build_feature_rows output is overwritten ONLY in the routed trigger columns of a modified COPY of the service (fill_value = finite synthetic value,
    NaN = missing), then the PUBLIC predict_batch / predict_baseline_batch run through the unchanged runtime path (same 19 schema/order, same pinned
    20 bundles incl. their preprocessors, same 20 context_complete/routed_prediction). Expected values come from the designed row kinds, not
    from the routing mask. Returns (rows, summary); every row is labelled SYNTHETIC_EVIDENCE and must never be reported as observed evidence."""
    S, H, c20 = service['units'][unit], service['H'], service['cfg20']
    trig, kinds = list(S['trigger']), SYNTHETIC_KINDS
    real_build = H['19']['build_feature_rows']

    def synth_build(ctx, R, u, variant):
        X, M = real_build(ctx, R, u, variant)
        X = X.copy()
        for i, k in enumerate(kinds[:len(X)]):
            X.iloc[i, [X.columns.get_loc(c) for c in trig]] = np.nan if k == 'all_missing' else fill_value
            if k in ('missing_first_trigger', 'missing_last_trigger'):
                X.iloc[i, X.columns.get_loc(trig[0] if k == 'missing_first_trigger' else trig[-1])] = np.nan
        return X, M

    h19 = types.MappingProxyType(dict(H['19'], build_feature_rows=synth_build))
    synth = types.MappingProxyType(dict(service, H=types.MappingProxyType(dict(H, **{'19': h19}))))
    segs = [str(s) for s in service['seg_index'][:len(kinds)]]
    reqs = [{'segment_id': s, 'target_date': '2024-03-01', **({'time_block': S['blocks'][0]} if unit == '6h' else {})} for s in segs]
    rs, rb, rb_real = predict_batch(synth, unit, reqs), predict_baseline_batch(synth, unit, reqs), predict_baseline_batch(service, unit, reqs)
    require(rs['ok'] and rb['ok'] and rb_real['ok'], 'synthetic routing probe request failed: ' + str(rs.get('error') or rb.get('error')))
    X, _ = synth_build(service['ctx'], validate_public(service, unit, reqs), unit, 'calendar_geometry_context')
    schema_ok = list(X.columns) == list(S['ctx_cols']) and list(X.columns[:len(S['core_cols'])]) == list(S['core_cols'])
    pc = H['20']['predict_component'](S['est']['core'], X, list(S['core_cols']), c20)
    px = H['20']['predict_component'](S['est']['context'], X, list(S['ctx_cols']), c20)
    rows = []
    for i, (k, out, ob, obr) in enumerate(zip(kinds, rs['results'], rb['results'], rb_real['results'])):
        want_comp = 'context' if k == 'context_complete' else 'core'
        want_reason = {'context_complete': 'context_complete', 'missing_first_trigger': 'core_fallback_missing_context:' + trig[0],
                       'missing_last_trigger': 'core_fallback_missing_context:' + trig[-1],
                       'all_missing': 'core_fallback_missing_context:' + ','.join(trig)}[k]
        want_lam = float(px[i] if want_comp == 'context' else pc[i])
        err = abs(out['expected_recorded_count'] - want_lam)
        rows.append({'evidence': SYNTHETIC_EVIDENCE, 'unit': unit, 'row': i, 'kind': k, 'component_used': out['component_used'], 'expected_component': want_comp,
                     'routing_reason': out['routing_reason'], 'expected_reason': want_reason, 'expected_recorded_count': out['expected_recorded_count'],
                     'core_prediction': float(pc[i]), 'context_prediction': float(px[i]), 'abs_error': err,
                     'baseline_component': ob['component_used'], 'baseline_reason': ob['routing_reason'],
                     'baseline_unchanged_by_context': ob['expected_recorded_count'] == obr['expected_recorded_count'],
                     'passed': bool(out['component_used'] == want_comp and out['routing_reason'] == want_reason and err <= 1e-12 and schema_ok
                                    and ob['component_used'] == 'baseline' and ob['routing_reason'] == 'baseline_has_no_routing'
                                    and ob['expected_recorded_count'] == obr['expected_recorded_count'])})
    ctx_rows = [r for r in rows if r['expected_component'] == 'context']
    summary = {'evidence': SYNTHETIC_EVIDENCE, 'unit': unit, 'rows': len(rows), 'context_rows': len(ctx_rows), 'core_rows': len(rows) - len(ctx_rows),
               'schema_and_order_equal_pinned_bundle_columns': schema_ok, 'fill_value': fill_value,
               'context_differs_from_core_on_context_rows': any(r['context_prediction'] != r['core_prediction'] for r in ctx_rows),
               'all_passed': all(r['passed'] for r in rows)}
    return rows, summary


def tamper_load_cases(backend, cfg, pins, baseline_files, out_dir):
    """load_service with modified COPIES of the trusted pins / baseline record / an on-disk bundle copy: each must be refused explicitly."""
    import copy
    out = []

    def case(name, p=None, bf=None, want=('artifact_hash_mismatch', 'artifact_missing', 'artifact_unavailable', 'artifact_inconsistent')):
        s = load_service(backend, cfg, p if p is not None else pins, bf if bf is not None else baseline_files, selftest=False)
        code = s['error']['code'] if s['error'] else None
        out.append({'case': name, 'status': s['status'], 'got_code': code, 'passed': s['status'] == 'unavailable' and code in want,
                    'message': (s['error'] or {}).get('message', '')[:200]})
    k = lambda suffix: next(x for x in pins['files'] if x.endswith(suffix))
    for name, suffix in (('tampered_source20_pin', '20_spatial_count_experiments.ipynb'), ('tampered_config20_pin', 'spatial_experiment_v3.json'),
                         ('tampered_model_bundles_pin', 'model_bundles.json'), ('tampered_selection_pin', 'selection_frozen.json'),
                         ('tampered_colour_threshold_pin', 'colour_contract_6h.json'), ('tampered_context19_manifest_pin', 'spatial_preparation_manifest.json')):
        p = copy.deepcopy(pins)
        p['files'][k(suffix)] = '0' * 64
        case(name, p)
    p = copy.deepcopy(pins)
    p['files']['backend/data/v3/spatial_serving/does_not_exist.json'] = '0' * 64
    case('missing_pinned_file', p, want=('artifact_missing',))
    p = copy.deepcopy(pins)
    p['functions_sha256']['notebook19'] = '0' * 64
    case('tampered_nb19_functions_pin', p, want=('artifact_inconsistent', 'artifact_hash_mismatch'))
    p = copy.deepcopy(pins)
    p['selection']['daily'] = 'hgb_poisson_fixed::context'
    case('pure_context_ablation_refused', p, want=('artifact_inconsistent',))
    p = copy.deepcopy(pins)
    p['kind'] = 'something_else'
    case('wrong_pins_kind', p, want=('artifact_unavailable',))
    bf = {u: (v[0], '0' * 64) for u, v in baseline_files.items()}
    case('tampered_baseline_stats_sha', bf=bf)
    # on-disk tampered bundle COPY (accepted files are never modified): the 20 loader must refuse before joblib.load
    docs, _ = verify_chain(backend, cfg, pins)
    H = load_helpers(backend, cfg, pins, docs)
    rel = docs['files20']['daily:core.joblib']
    src = docs['paths']['dat20'] / rel
    d = Path(out_dir) / 'tamper_fixture'
    d.mkdir(parents=True, exist_ok=True)
    b = bytearray(src.read_bytes())
    b[len(b) // 2] ^= 0xFF
    (d / src.name).write_bytes(bytes(b))
    (d / src.with_suffix('.meta.json').name).write_bytes(src.with_suffix('.meta.json').read_bytes())
    try:
        H['20']['load_bundle'](d / src.name, docs['man20']['data_files_gitignored'][rel])
        refused = False
    except Exception as e:
        refused = 'sha256' in str(e) or 'differs' in str(e)
    out.append({'case': 'tampered_bundle_bytes_copy', 'status': 'refused' if refused else 'LOADED', 'got_code': 'artifact_hash_mismatch' if refused else None,
                'passed': refused, 'message': 'flipped one byte of a copy; load_bundle refused before joblib.load' if refused else 'loaded a tampered bundle'})
    return out


def csv_reload_parity(df, path, H):
    atomic_write_bytes(path, csv_bytes(df))
    back = H['19']['read_numeric_csv'](path, ['segment_id', 'target_date', 'time_block', 'which', 'unit', 'component_used', 'routing_reason', 'colour', 'bank_calendar'])
    a, b = df['expected_recorded_count'].to_numpy(np.float64), back['expected_recorded_count'].to_numpy(np.float64)
    return bool(len(a) == len(b) and np.array_equal(a, b)), sha256_file(path)


def latency(service, unit, refk, cfg):
    L, need, rows = cfg['serving']['latency'], need_fields(unit), []
    seen = refk[refk['diag_offset'] >= 0]
    for bs in L['batch_sizes']:
        reqs = [{k: str(r[k]) for k in need} for _, r in seen.head(int(bs)).iterrows()]
        require(len(reqs) == int(bs), 'not enough reference requests for latency batch ' + str(bs))
        for which, fn in (('selected', predict_batch), ('baseline', predict_baseline_batch)):
            for _ in range(int(L['warmup'])):
                fn(service, unit, reqs)
            t = []
            for _ in range(int(L['repetitions'])):
                t0 = time.perf_counter()
                r = fn(service, unit, reqs)
                t.append((time.perf_counter() - t0) * 1000.0)
                require(r['ok'], 'latency request failed')
            t = np.asarray(t)
            rows.append({'unit': unit, 'which': which, 'batch_size': int(bs), 'warmup': int(L['warmup']), 'repetitions': int(L['repetitions']),
                         'ms_min': float(t.min()), 'ms_median': float(np.median(t)), 'ms_p95': float(np.quantile(t, 0.95)), 'ms_max': float(t.max()),
                         'note': L['note']})
    return rows


In [ ]:
# FUNCTIONS (5/5): Thai readiness report generated ONLY from computed results, and the strict manifest (checks, input/output hashes,
# protected before/after). research_ready may be true while production_ready and serving_deployed stay false.
def _f(v, nd=4):
    return 'n/a' if v is None or (isinstance(v, float) and not math.isfinite(v)) else (('{:.' + str(nd) + 'g}').format(v) if isinstance(v, float) else str(v))


def readiness_report_th(meta, parity, selftests, cases, tampers, lat, ref_summary, checks, synthetic=None):
    failed = [c['check'] for c in checks if c['status'] != 'pass']
    L = ['# รายงานความพร้อมเชิงวิจัยของ runtime การพยากรณ์เชิงพื้นที่ (V3, notebook 22)', '',
         '**สิ่งที่พยากรณ์:** จำนวนบันทึกที่คาดหมาย (lambda) ของบันทึก SELECTED+LINKED ต่อช่วงถนน x วัน หรือ x ช่วง 6 ชั่วโมง '
         'ไม่ใช่ความน่าจะเป็นเกิดอุบัติเหตุรายบุคคล ไม่ใช่ความเสี่ยงต่อยานพาหนะ และห้ามรวมเป็นความน่าจะเป็นของเส้นทาง', '',
         '- เวอร์ชันโมเดล: `' + meta['model_version'] + '` (benchmark, ออก ณ ' + meta['fit_issue_local'] + ', ใช้ label ถึง ' + meta['training_label_cutoff_local'] + ')',
         '- ช่วงวันที่เป้าหมายที่รองรับ: ' + meta['supported_target_start'] + ' ถึง ' + meta['supported_target_end'] +
         ' (หลัง ' + meta['observed_diagnostic_end'] + ' เป็น research scenario ไม่มีค่าจริงให้เทียบ)',
         '- ไม่มี extended refit และไม่มีโมเดล OOF fold จึงไม่อ้างการ replay OOF', '',
         '## ความตรงกันกับผล notebook 20 ที่บันทึกไว้ (ทั้งประชากร 2024/25 ที่เคยเห็นแล้ว)', '']
    for r in parity:
        L.append('- ' + r['unit'] + ' `' + r['method'] + '`: ' + str(r['rows']) + ' แถว, max abs error ' + _f(r['max_abs_error'], 3) + ', เกิน tolerance ' +
                 str(r['cells_over_tolerance']) + ' แถว, routing mask ไม่ตรง ' + str(r['routing_mask_mismatches']) + ' แถว; ขอบเขตจริงจาก mask ที่บันทึกไว้: '
                 + str(r.get('observed_routing_scope')) + ' (context ' + str(r['context_rows']) + ' / core ' + str(r['core_rows']) + ')')
    if synthetic:
        L += ['', '## ตรวจการ routing ด้วยข้อมูลสังเคราะห์ (SYNTHETIC fixture — ไม่ใช่หลักฐานจากข้อมูลจริงปี 2024/25)', '']
        L += ['- ' + s['unit'] + ': ' + str(s['rows']) + ' แถวสังเคราะห์ (context ' + str(s['context_rows']) + ' / core ' + str(s['core_rows']) + '), ผ่านทั้งหมด: '
              + str(s['all_passed']) for s in synthetic]
    L += ['', '## ตรวจตัวแบบด้วย reference requests ภายใน (รวมปี 2021 จึงไม่ใช่การพยากรณ์ล่วงหน้า)', '']
    L += ['- ' + r['unit'] + ' ' + r['component'] + ': ' + str(r['rows']) + ' แถว, max abs error ' + _f(r['max_abs_error'], 3) for r in selftests]
    L += ['', '## คำขอสาธารณะที่ติดตามได้', '']
    L += ['- ' + k + ': ' + _f(v) for k, v in ref_summary.items()]
    L += ['', '## ความทนทาน', '', '- กรณีคำขอผิด/นอกขอบเขต/ผลไม่ถูกต้อง ผ่าน ' + str(sum(c['passed'] for c in cases)) + '/' + str(len(cases)),
          '- กรณีไฟล์ถูกแก้/ขาดหาย ถูกปฏิเสธอย่างชัดเจน ' + str(sum(c['passed'] for c in tampers)) + '/' + str(len(tampers)), '',
          '## เวลาตอบสนองที่วัดจริง (ไม่ใช่ SLA)', '']
    L += ['- ' + r['unit'] + ' ' + r['which'] + ' batch ' + str(r['batch_size']) + ': median ' + _f(r['ms_median'], 3) + ' ms, p95 ' + _f(r['ms_p95'], 3) + ' ms'
          for r in lat]
    L += ['', '## สถานะ', '', '- research_ready: **' + str(not failed) + '**', '- production_ready: **False** (ยังไม่ deploy, ยังไม่มี API019)',
          '- checks ที่ไม่ผ่าน: ' + (', '.join(failed) if failed else 'ไม่มี'), '', '## แหล่งข้อมูลและข้อจำกัด', '']
    L += ['- ' + k + ': ' + v for k, v in meta['provenance'].items()] + ['- ' + c for c in meta['caveats']]
    return NL.join(L) + NL


def build_manifest(status, started, cfg, cfg_sha, pins_sha, checks, report_files, data_files, protected_before, protected_after, extra):
    changed = sorted(k for k in set(protected_before) | set(protected_after) if protected_before.get(k) != protected_after.get(k))
    failed = [c['check'] for c in checks if c['status'] != 'pass']
    return {'status': status, 'version': cfg['version'], 'purpose': 'verified research inference + robustness contract of the accepted benchmark bundles; NOT deployed',
            'label_meaning': cfg['label_meaning'], 'started_utc': started, 'finished_utc': utcnow(), 'config_sha256': cfg_sha, 'pins_sha256': pins_sha,
            'notebook': cfg['notebook'], 'environment': environment(), 'checks': checks, 'failed_checks': failed, 'report_files': report_files,
            'data_files_gitignored': data_files, 'protected_files_hashed': len(protected_before), 'protected_unchanged': not changed, 'protected_changed': changed,
            'protected_sha256_before': protected_before,
            'readiness': {'research_ready': status == 'complete' and not failed, 'production_ready': False, 'serving_deployed': False,
                          'default_api_artifact': cfg['artifact']['model_version'], 'caveats': cfg['caveats']}, **extra}


In [ ]:
# SETUP: config, dependency check, NEW output root guard (before any write), protected snapshot, check/stage helpers.
import contextlib
import resource

STARTED = utcnow()
cfg = read_strict_json(CONFIG_PATH)
CFG_SHA = sha256_file(CONFIG_PATH)
VERSIONS = require_modules(cfg['dependency_pins'])
PATHS = output_paths(OUTPUT_ROOT, cfg)
check_outputs(PATHS, BACKEND, cfg)
PROTECTED_BEFORE = hash_paths(BACKEND, protected_rels(cfg))
PATHS['report_dir'].mkdir(parents=True)
PATHS['data_dir'].mkdir(parents=True)
CHECKS, DURATIONS, REPORT_FILES, T = [], {}, {}, {}


def check(name, ok, evidence=None):
    CHECKS.append({'check': name, 'status': 'pass' if ok else 'fail', 'evidence': jsonable(evidence)})
    if not ok:
        raise ServingError('artifact_inconsistent', 'check failed: ' + name + ' | ' + json.dumps(jsonable(evidence), allow_nan=False)[:600])


def peak_rss_bytes():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return int(r if platform.system() == 'Darwin' else r * 1024)


@contextlib.contextmanager
def stage(name):
    t0 = time.time()
    print('stage', name, '...', flush=True)
    yield
    DURATIONS[name] = round(time.time() - t0, 2)
    print('stage', name, 'done in', DURATIONS[name], 's', flush=True)


def put_report(name, obj):
    check_report_file(name, cfg)
    b = csv_bytes(obj) if isinstance(obj, pd.DataFrame) else (obj if isinstance(obj, bytes) else strict_json_bytes(obj))
    atomic_write_bytes(PATHS['report_dir'] / name, b)
    REPORT_FILES[name] = sha256_bytes(b)
    return REPORT_FILES[name]


print('versions', VERSIONS, '| report dir', PATHS['report_dir'], '| data dir', PATHS['data_dir'])


In [ ]:
# INPUTS + BASELINE STATS: trusted manager pins -> full chain verification -> hash-verified 19/20/21 FUNCTIONS -> pinned prepared context and labels
# -> the ONE offline reconstruction of the segment_mean_k365 sufficient statistics with the exact 20 benchmark cutoff/recipe (no model fit).
with stage('inputs_and_baseline_stats'):
    PINS_PATH = BACKEND / cfg['pins']['path']
    PINS_SHA = sha256_file(PINS_PATH)
    PINS = read_strict_json(PINS_PATH)
    DOCS, INPUT_ROWS = verify_chain(BACKEND, cfg, PINS)
    check('manager pins, 19/20/21 producer manifests and every consumed artifact hash-verified before any source exec or deserialization', True,
          {'verified_files': len(INPUT_ROWS), 'pins_sha256': PINS_SHA})
    HLP = load_helpers(BACKEND, cfg, PINS, DOCS)
    HLP['cfg20'] = DOCS['cfg20']
    check('19/21 FUNCTIONS equal the manager pins; 19 code cells equal the 20 lock; 20 source notebook file pinned', True, HLP['functions_sha256'])
    ISSUE, CUTOFF = benchmark_times(HLP, cfg)
    check('declared artifact as-of issue and training label cutoff equal the pinned 20 recipe', True, {'issue': str(ISSUE), 'cutoff': str(CUTOFF)})
    n19 = HLP['19']
    CTX = n19['load_prepared_context'](DOCS['paths']['prepared'], DOCS['man19']['data_files_gitignored'])
    AXJ = read_strict_json(DOCS['paths']['prepared'] / 'population_axes.json')
    AX = n19['population_axes'](AXJ['segments'], AXJ['dates'][0], AXJ['dates'][-1], AXJ['blocks'])
    check('population axes = 19 axes file = sorted universe (3105 qualified DOH segments)', AX['segments'] == sorted(CTX['seg'].index) == AXJ['segments'],
          {'segments': len(AX['segments']), 'dates': [AX['dates'][0], AX['dates'][-1]]})
    for name in ('counts_daily.csv.gz', 'counts_6h.csv.gz', 'population_axes.json'):
        verify_file(DOCS['paths']['prepared'] / name, DOCS['man19']['data_files_gitignored'].get(name), 'prepared ' + name)
    COUNTS = {'daily': n19['read_numeric_csv'](DOCS['paths']['prepared'] / 'counts_daily.csv.gz', ['segment_id', 'target_date']),
              '6h': n19['read_numeric_csv'](DOCS['paths']['prepared'] / 'counts_6h.csv.gz', ['segment_id', 'target_date', 'time_block'])}
    SP = {u: n19['sparse_index'](AX, u, COUNTS[u]) for u in UNITS}
    BASE = PINS['selection']['baseline']
    BASELINE_FILES, BASE_EVID = {}, []
    for u in UNITS:
        st, ev = build_baseline_stats(HLP, cfg, CTX, AX, SP[u], u, BASE)
        p = PATHS['data_dir'] / ('baseline_stats_' + u + '.npz')
        BASELINE_FILES[u] = (p, save_baseline_stats(p, st))
        BASE_EVID.append(dict(ev, baseline=BASE, sha256=BASELINE_FILES[u][1], cutoff_local=str(CUTOFF)))
    del COUNTS, SP
    T['baseline_stats'] = pd.DataFrame(BASE_EVID)
    check('baseline sufficient statistics rebuilt once with the exact 20 benchmark cutoff (no fallback cells unexpectedly)', all(e['unseen_segments'] == 0 for e in BASE_EVID), BASE_EVID)
T['baseline_stats']


In [ ]:
# SERVICE + PARITY: a fresh load_service re-verifies everything from disk (it shares nothing with the cells above except the trusted baseline-stats
# record), then the FULL saved 2024/25 diagnostic populations are streamed through the same predict path and compared with the notebook-20 arrays.
with stage('service_and_parity'):
    SERVICE = load_service(BACKEND, cfg, PINS, BASELINE_FILES)
    HEALTH = health(SERVICE)
    check('service startup available; production_ready reported false separately', SERVICE['status'] == 'available' and HEALTH['production_ready'] is False, HEALTH)
    T['selftest'] = pd.DataFrame(selftest_rows(SERVICE))
    check('bundle reference requests (internal, incl. 2021) reproduce the saved 20 component predictions within tolerance', bool(T['selftest']['pass'].all()),
          T['selftest'].to_dict('records'))
    PARITY, RUNTIME_ARRAYS = [], {}
    for u in UNITS:
        rows, meta = full_parity(SERVICE, DOCS, AX, u, PATHS['data_dir'] / 'parity')
        PARITY += rows
        RUNTIME_ARRAYS[u] = meta
    T['parity'] = pd.DataFrame(PARITY)
    tol = float(cfg['serving']['parity_abs_tolerance'])
    check('runtime reproduces saved 20 selected-policy AND baseline predictions on the full 2024/25 populations (max abs error <= 1e-12)',
          all(r['max_abs_error'] <= tol and r['cells_over_tolerance'] == 0 and r['rows'] == r['population_rows'] for r in PARITY),
          [{k: r[k] for k in ('unit', 'method', 'rows', 'max_abs_error')} for r in PARITY])
    check('runtime routing mask equals the saved 20 context mask on every cell', all(r['routing_mask_mismatches'] == 0 for r in PARITY))
    # observed scope is REPORTED from the saved mask (2024/25 may have no observed weather/AADT context -> core-only); context routing itself is
    # exercised separately by the clearly synthetic probe in the robustness stage, never presented as observed evidence
    check('observed routing scope reported from the saved 20 mask (context rows == saved mask ones; context + core == rows)',
          all(r['context_rows'] == r['saved_mask_context_rows'] and r['context_rows'] + r['core_rows'] == r['rows'] for r in PARITY),
          [{k: r[k] for k in ('unit', 'context_rows', 'core_rows', 'saved_mask_context_rows', 'observed_routing_scope')} for r in PARITY])
T['parity'][['unit', 'method', 'rows', 'max_abs_error', 'cells_over_tolerance', 'routing_mask_mismatches', 'context_rows', 'core_rows', 'observed_routing_scope', 'seconds']]


In [ ]:
# PUBLIC REQUESTS + ROBUSTNESS + LATENCY: traced deterministic requests (seen rows checked against saved 20 arrays by offset; 2026-27 rows are
# research scenarios), request/coverage/prediction guards, tampered/missing artifact loads, CSV reload parity, finite JSON, measured latency.
with stage('requests_robustness_latency'):
    REFK, REF, FIXTURES, CASES, LAT, SYNTH, SYNTH_SUMMARY = {}, [], {}, [], [], [], []
    for u in UNITS:
        REFK[u] = reference_requests(SERVICE, AX, DOCS, u, cfg)
        df, responses = run_reference(SERVICE, DOCS, u, REFK[u])
        REF.append(df)
        FIXTURES[u] = {'requests': REFK[u].to_dict('records'), 'responses': responses}
        CASES += [dict(c, unit=u) for c in run_cases(robustness_cases(SERVICE, u))]
        LAT += latency(SERVICE, u, REFK[u], cfg)
        rows_, summ_ = synthetic_routing_probe(SERVICE, u)
        SYNTH += rows_
        SYNTH_SUMMARY.append(summ_)
    T['reference'] = pd.concat(REF, ignore_index=True)
    R_ = T['reference']
    seen, fut = R_[R_['diag_offset'] >= 0], R_[R_['diag_offset'] < 0]
    tol = float(cfg['serving']['parity_abs_tolerance'])
    ssel = seen[seen['which'] == 'selected']
    REF_SUMMARY = {'rows': len(R_), 'seen_rows_traced_to_20_arrays': len(seen), 'research_scenario_rows': len(fut),
                   'max_abs_error_seen': float(seen['abs_error'].max()), 'context_rows': int((R_['component_used'] == 'context').sum()),
                   'core_rows': int((R_['component_used'] == 'core').sum()), 'future_core_rows': int(((fut['which'] == 'selected') & (fut['component_used'] == 'core')).sum()),
                   'seen_selected_saved_mask_context_rows': int((ssel['saved_context_mask'] == 1).sum()),
                   'observed_routing_scope_seen': routing_scope(int((ssel['saved_context_mask'] == 1).sum()), len(ssel)),
                   'bank_calendar_unknown_rows': int((R_['bank_calendar'] != 'known').sum())}
    check('public predict_batch / predict_baseline_batch reproduce saved 20 values at traced offsets', REF_SUMMARY['max_abs_error_seen'] <= tol, REF_SUMMARY)
    check('2026-27 rows are flagged research scenarios and 2024-25 rows are not', bool(fut['research_scenario'].all() and not seen['research_scenario'].any()))
    sel_rows = R_[R_['which'] == 'selected']
    check('seen selected components equal the saved 20 mask at their offsets (scope reported, not assumed); context iff reason context_complete; '
          'baseline rows report the baseline component only',
          bool((ssel['component_used'] == np.where(ssel['saved_context_mask'] == 1, 'context', 'core')).all()
               and ((sel_rows['component_used'] == 'context') == (sel_rows['routing_reason'] == 'context_complete')).all()
               and (R_[R_['which'] == 'baseline']['component_used'] == 'baseline').all()), REF_SUMMARY)
    T['synthetic_routing'] = pd.DataFrame(SYNTH)
    check('SYNTHETIC fixture (not observed evidence): context-complete / core-fallback / mixed batches route through the runtime path with the pinned '
          'schema and bundles; outputs, components and reasons match the designed rows; baseline unaffected',
          bool(T['synthetic_routing']['passed'].all()) and all(s['all_passed'] and s['context_rows'] > 0 and s['core_rows'] > 0 and
                                                               s['schema_and_order_equal_pinned_bundle_columns'] and s['context_differs_from_core_on_context_rows']
                                                               for s in SYNTH_SUMMARY), SYNTH_SUMMARY)
    check('core fallback reason always names the missing context columns', bool(R_[(R_['which'] == 'selected') & (R_['component_used'] == 'core')]['routing_reason']
                                                                                 .str.startswith('core_fallback_missing_context:').all()))
    check('selected rows always carry a colour; "safe"/green never appears', bool(R_[R_['which'] == 'selected']['colour'].isin(['base', 'yellow', 'orange', 'red']).all()))
    T['cases'] = pd.DataFrame(CASES)
    check('request/coverage/prediction robustness cases return the expected explicit codes', bool(T['cases']['passed'].all()),
          T['cases'][~T['cases']['passed']].to_dict('records'))
    TAMPER = tamper_load_cases(BACKEND, cfg, PINS, BASELINE_FILES, PATHS['data_dir'])
    T['tamper'] = pd.DataFrame(TAMPER)
    check('missing/tampered source, config, model, selection, colour threshold, context manifest and baseline stats are refused explicitly',
          bool(T['tamper']['passed'].all()), T['tamper'][~T['tamper']['passed']].to_dict('records'))
    ok_csv, CSV_SHA = csv_reload_parity(R_, PATHS['data_dir'] / 'reference_responses.csv', SERVICE['H'])
    check('reference responses reload from CSV bit-identically (round-trip floats)', ok_csv, CSV_SHA)
    fx = strict_json_bytes(FIXTURES)
    json.loads(fx.decode('utf-8'), parse_constant=lambda c: (_ for _ in ()).throw(ValueError(c)))
    atomic_write_bytes(PATHS['data_dir'] / 'prediction_fixtures.json', fx)
    check('every fixture/response is strict finite JSON', True, {'bytes': len(fx)})
    T['latency'] = pd.DataFrame(LAT)
    check('latency measured for batch 1 and 128 with warm repetitions (actual timing, no SLA)', set(T['latency']['batch_size']) == set(cfg['serving']['latency']['batch_sizes']))
T['latency']


In [ ]:
# REPORTS + MANIFEST: aggregate CSV/JSON/Thai readiness report (row-level arrays, fixtures and stats stay in the ignored data dir), the serving
# record a later API pins (baseline-stats hashes, bundle hashes, interfaces), protected before/after and the strict manifest.
with stage('reports'):
    for name, df in (('parity_full_population.csv', T['parity']), ('bundle_reference_selftest.csv', T['selftest']), ('baseline_stats.csv', T['baseline_stats'].astype(str)),
                     ('robustness_cases.csv', T['cases']), ('tamper_cases.csv', T['tamper']), ('latency.csv', T['latency']),
                     ('synthetic_routing_fixture_NOT_observed.csv', T['synthetic_routing']),
                     ('reference_requests_summary.csv', T['reference'].groupby(['unit', 'which', 'component_used', 'research_scenario']).agg(
                         rows=('expected_recorded_count', 'size'), max_abs_error=('abs_error', 'max'), mean_lambda=('expected_recorded_count', 'mean')).reset_index()),
                     ('inputs.csv', pd.DataFrame(INPUT_ROWS))):
        put_report(name, df)
    SERVING_RECORD = {'kind': 'roadaware_spatial_serving_record', 'version': cfg['version'], 'model_version': cfg['artifact']['model_version'],
                      'default_for_api': cfg['artifact']['default_for_api'], 'pins_sha256': PINS_SHA, 'config_sha256': CFG_SHA,
                      'baseline_stats': {u: {'file': Path(p).name, 'sha256': s} for u, (p, s) in BASELINE_FILES.items()},
                      'metadata': metadata(SERVICE), 'health': HEALTH,
                      'interfaces': {'load_service(backend, cfg, pins, baseline_files)': 'trusted server-side inputs only; returns an immutable available/unavailable service',
                                     'predict_batch(service, unit, requests)': 'selected policy, 1..128 rows, ordered output, strict finite JSON',
                                     'predict_baseline_batch(service, unit, requests)': 'locked segment_mean_k365, same keys/guards',
                                     'metadata(service)': 'artifact facts, fields, cutoffs, hashes, error codes', 'coverage(service, segment_ids)': 'supported/unsupported/unknown',
                                     'health(service)': 'startup availability separate from production_ready=false'}}
    put_report('serving_record.json', SERVING_RECORD)
    STATUS = 'complete'
    put_report('readiness_th.md', readiness_report_th(dict(SERVICE['meta']), PARITY, T['selftest'].to_dict('records'), CASES, TAMPER, LAT, REF_SUMMARY, CHECKS, SYNTH_SUMMARY).encode('utf-8'))
    DATA_FILES = {q.relative_to(PATHS['data_dir']).as_posix(): sha256_file(q) for q in sorted(PATHS['data_dir'].rglob('*')) if q.is_file()}
    PROTECTED_AFTER = hash_paths(BACKEND, protected_rels(cfg))
    check('protected V2/V3 inputs, accepted 19/20/21 outputs, configs, notebooks, pins unchanged', PROTECTED_AFTER == PROTECTED_BEFORE, {'files': len(PROTECTED_BEFORE)})
    check('source config and manager pins unchanged during the run', sha256_file(CONFIG_PATH) == CFG_SHA and sha256_file(PINS_PATH) == PINS_SHA)
    EXTRA = {'pins': PINS, 'functions_sha256': HLP['functions_sha256'], 'experiment_lock_sha256': DOCS['lock_sha'], 'selection_sha256': DOCS['sel_sha'],
             'artifact': cfg['artifact'], 'parity': PARITY, 'runtime_arrays': RUNTIME_ARRAYS, 'reference_summary': REF_SUMMARY, 'synthetic_routing_fixture_summary': SYNTH_SUMMARY,
             'baseline_stats': SERVING_RECORD['baseline_stats'], 'reference_csv_sha256': CSV_SHA, 'durations': DURATIONS, 'peak_rss_bytes': peak_rss_bytes()}
    MANIFEST = build_manifest(STATUS, STARTED, cfg, CFG_SHA, PINS_SHA, CHECKS, REPORT_FILES, DATA_FILES, PROTECTED_BEFORE, PROTECTED_AFTER, EXTRA)
    atomic_write_bytes(PATHS['report_dir'] / 'spatial_serving_manifest.json', strict_json_bytes(MANIFEST))
    read_strict_json(PATHS['report_dir'] / 'spatial_serving_manifest.json')
print('status', MANIFEST['status'], '| failed checks', MANIFEST['failed_checks'], '| research_ready', MANIFEST['readiness']['research_ready'], '| production_ready False')


## Outputs

* `reports/v3/spatial_serving/` (aggregate): `spatial_serving_manifest.json` (written last, only on full success), `serving_record.json`,
  `parity_full_population.csv`, `bundle_reference_selftest.csv`, `baseline_stats.csv`, `reference_requests_summary.csv`, `robustness_cases.csv`,
  `tamper_cases.csv`, `latency.csv`, `synthetic_routing_fixture_NOT_observed.csv` (SYNTHETIC routing check, not observed evidence), `inputs.csv`, `readiness_th.md`.
* Observed 2024/25 routing scope is reported from the saved 20 mask (`observed_routing_scope` in the parity CSV; core-only when no weather/AADT context was observed).
* `data/v3/spatial_serving/` (gitignored): `baseline_stats_{daily,6h}.npz`, `parity/runtime_diag_<unit>_{selected,baseline,context_mask}.npy`,
  `prediction_fixtures.json`, `reference_responses.csv`, `tamper_fixture/` (a byte-flipped COPY of one bundle; never loaded).
* Independent check: `.venv/bin/python backend/scripts/verify_spatial_inference.py --run-root backend/rerun-spatial-serving-<id>`.
* Research readiness only. `production_ready` is false; nothing is deployed.
